# Part 3 — Activation-Space Safety Vector (Function Vector)

Causal Mediation Analysis over all layer×head pairs, top-10 heads by Average Indirect Effect, and injection of the resulting Function Vector into the residual stream at layer `floor(L/3)`.

In [ ]:
# --- environment -----------------------------------------------------------
import os, sys, glob, shutil, zipfile
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"                  # dataset + model cache
os.environ["SAFEALIGN_ROOT"] = "/kaggle/temp/safealign"    # big artifacts, off the 20 GB quota
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:                                                        # Add-ons -> Secrets -> HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not found:", e)

!pip install -q -U "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" \
    rouge-score sacrebleu nltk mergekit

# --- locate the project inside the attached Kaggle Dataset -----------------
REPO = "/kaggle/working/safety-alignment-llm"

def find_source():
    # Kaggle auto-extracts uploaded archives, so the dataset may hold either
    # the unpacked folder or the original .zip. Handle both.
    hits = glob.glob("/kaggle/input/**/src/safealign/config.py", recursive=True)
    if hits:
        return ("dir", str(Path(hits[0]).parents[2]))
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if zips:
        return ("zip", zips[0])
    raise FileNotFoundError("Attach the dataset holding the project (Add Input -> Datasets)")

if not os.path.exists(REPO):
    kind, src = find_source()
    if kind == "zip":
        with zipfile.ZipFile(src) as z:
            z.extractall("/kaggle/working")
    else:
        shutil.copytree(src, REPO)                          # /kaggle/input is read-only
    print("project from", kind, src)

sys.path.insert(0, f"{REPO}/src")

import torch
print(torch.__version__, "| GPUs:", torch.cuda.device_count(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
from safealign.config import CFG
CFG.paths.ensure(); print("artifacts ->", CFG.paths.artifacts)


## 3.1 Few-shot prompts and the sampling strategy

541 toxic-dpo rows are shuffled once under a fixed seed and split into a 15-row target pool and a 526-row in-context pool. The pools are disjoint, so no target query can appear among its own exemplars. Each prompt draws N=10 exemplars without replacement from the ICL pool under an independent seed.

In [ ]:
from safealign.fv.prompts import build_fewshot_pairs, refusal_token_ids
from safealign.model_utils import load_tokenizer

pairs = build_fewshot_pairs()
tok = load_tokenizer()
print('prompts:', len(pairs), 'ICL per prompt:', len(pairs[0].icl_indices))
print('V_refusal:', [tok.decode([i]) for i in refusal_token_ids(tok)])
print(pairs[0].clean_messages[1]['content'][:200])
print('CLEAN  ->', pairs[0].clean_messages[2]['content'][:160])
print('CORRUPT->', pairs[0].corrupted_messages[2]['content'][:160])

## Causal Indirect Effect

Head activations are read **after** the per-head slice of `W_O`, so they live in residual-stream space. Heads are evaluated in batches of `head_batch_size`, one head per batch element, instead of one forward pass per head.

In [ ]:
from safealign.fv.cie import run_cma
cma = run_cma()
cma['top_heads']

## 3.2 Build and inject the Function Vector

In [ ]:
from safealign.fv.vector import build_and_save, sweep_lambda
from safealign.config import CFG, MODEL_SFT

fv_meta = build_and_save(); print(fv_meta)
lam = sweep_lambda(str(CFG.paths.artifacts / f'{MODEL_SFT}_merged'))
lam['best_lambda']

## 3.3 Interpretability

The AIE heatmap shows *where* in the network the safety computation happens; the logit lens shows whether the FV decodes to literal refusal words or to something more abstract.

In [ ]:
from safealign.fv.viz import build_all_figures
figs = build_all_figures(); figs['top_tokens']

In [ ]:
from IPython.display import Image, display
display(Image(figs['aie_heatmap'])); display(Image(figs['logit_lens_fig']))